In [18]:
import re
from stemming.porter2 import stem
from collections import Counter
from math import log2

In [15]:
# Prepare the book content
with open("book.txt", "r") as file:
    content = file.read().lower()

punctuation = '.,?!:;\'"’‘“”*()[]_-'
content = re.sub(f"[{re.escape(punctuation)}]", " ", content)

In [ ]:
# Exercise 5
def prepare_word_list(words: list) -> Counter:
    with open("stop-words.txt", "r") as file:
        stop_words = file.read().split() # TODO: it doesn't make sense; remove stop words from 'words'
    words = [word for word in words if word not in stop_words]

    words = [stem(word) for word in words]

    pairs = Counter(words) # average-case complexity O(n)
    pairs = sorted(pairs.items(), key=lambda x: x[1], reverse=True)
    pairs = pairs[10:]

    return pairs

In [ ]:
# Exercise 5
pairs = prepare_word_list(content.split())

with open("word-counts.csv", "w") as file:
    file.write("\"weight\",\"word\",\"color\",\"url\"\n")
    for word, count in pairs:
        file.write(f"\"{count}\",\"{word}\",\"\",\"\"\n")

In [33]:
# Exercise 6
chapters = content.split("chapter")
chapters = chapters[1:] # first element is not a chapter
chapter_words = [prepare_word_list(ch.split()[1:]) for ch in chapters]
D = len(chapters)

word_document_occurances = Counter()
for chw in chapter_words:
    for w, _ in chw:
        word_document_occurances[w] = word_document_occurances.get(w, 0) + 1

chapter_words_weights = [{word :                                                     \
                            counter * log2(D / (1 + word_document_occurances[word])) \
                            for word, counter in chw} for chw in chapter_words]

# TODO: handle the problem with negatives and floats in weights
#       (e.g. by normalizing weights to 0-100 range)
#       finally print those dumm word clouds

ids_of_chapters_to_print = [0, 15, 39]
for i in ids_of_chapters_to_print:
    with open(f"chapter-{i+1}-word-counts.csv", "w") as file:
        file.write("\"weight\",\"word\",\"color\",\"url\"\n")
        for word in chapter_words_weights[i]:
            file.write(f"\"{chapter_words_weights[i][word]}\",\"{word}\",\"\",\"\"\n")

In [45]:
# Exercise 7
def fit_the_chapter(word: str, chapter_words_weights: list[dict]) -> list:
    fits = {i+1 : chapter_words_weights[i].get(word, float('-inf')) for i in range(len(chapter_words_weights))}
    fits = sorted(fits.items(), key=lambda x: x[1], reverse=True)
    return fits

In [46]:
# Exercise 7
print(fit_the_chapter("heaven", chapter_words_weights))

[(26, 4.64805336054301), (2, 3.7184426884344077), (15, 3.7184426884344077), (14, 2.788832016325806), (16, 2.788832016325806), (22, 2.788832016325806), (25, 2.788832016325806), (7, 1.8592213442172039), (17, 1.8592213442172039), (18, 1.8592213442172039), (24, 1.8592213442172039), (6, 0.9296106721086019), (9, 0.9296106721086019), (10, 0.9296106721086019), (12, 0.9296106721086019), (21, 0.9296106721086019), (23, 0.9296106721086019), (28, 0.9296106721086019), (30, 0.9296106721086019), (33, 0.9296106721086019), (1, -inf), (3, -inf), (4, -inf), (5, -inf), (8, -inf), (11, -inf), (13, -inf), (19, -inf), (20, -inf), (27, -inf), (29, -inf), (31, -inf), (32, -inf), (34, -inf), (35, -inf), (36, -inf), (37, -inf), (38, -inf), (39, -inf), (40, -inf)]


In [49]:
# Exercise 8
with open("stop-words.txt", "r") as file:
    stop_words = file.read().split()
words = [word for word in content.split() if word not in stop_words]